# Study 946 — Distribution is not Return — the teardown

The Fama-MacBeth cross-sectional slopes, the tercile spread decomposed into payout and price legs, the give-back ratio, the excess-of-cash race with a CAPM control, block-bootstrap CIs, the era cut, the sort-width / guard / universe robustness grid, the cost × borrow sweep, and the live synthetic control.

Every real-tape number is frozen from `docs/results.md` (fingerprints of the tapes' **returns** — `ac334d204e26` / `b94d79082051`; level fingerprints are not reproducible because `auto_adjust=True` back-adjusts the whole history on every re-fetch), held months 2013-11-30 → 2026-06-30, n = 152. As-of 2026-06-30.

**Read the three left-hand sides as two.** The payout is *defined* as the total/price gap, so `hml_price ≡ hml_total − hml_payout` (correlation 0.99995). The erosion *t* is therefore the payout-persistence *t* carried through a total-return null — arithmetic, not a second experiment.

In [1]:
R = {'start': '2013-11-30', 'end': '2026-06-30', 'n_months': 152, 'fp_tr': 'ac334d204e26', 'fp_pr': 'b94d79082051', 'hml_lo_ann': -5.67, 'hml_hi_ann': 1.25, 'ident_corr': 0.99995, 'n_funds': 15, 'xs_min': 6, 'xs_max': 15, 'fm_dist': 24.6, 'fm_dist_t': 11.27, 'fm_price': -28.1, 'fm_price_t': -3.84, 'fm_total': -3.5, 'fm_total_t': -0.48, 'dist_hi': 9.92, 'dist_lo': 2.87, 'dist_spread': 7.05, 'hml_d': 51.6, 'hml_d_t': 10.27, 'hml_d_lo': 42.4, 'hml_d_hi': 60.5, 'hml_p': -69.9, 'hml_p_t': -4.53, 'hml_p_lo': -100.5, 'hml_p_hi': -40.4, 'hml': -18.3, 'hml_t': -1.24, 'hml_lo': -47.2, 'hml_hi': 10.4, 'hml_pgt0': 0.102, 'giveback': 1.36, 'hi_sharpe': 0.737, 'hi_mean': 62.6, 'hi_vol': 10.2, 'hi_dd': -20.1, 'hi_cagr': 9.06, 'lo_sharpe': 0.731, 'lo_mean': 80.9, 'lo_vol': 13.3, 'lo_dd': -23.3, 'lo_cagr': 11.07, 'spy_sharpe': 0.862, 'spy_mean': 104.4, 'spy_vol': 14.5, 'spy_dd': -24.4, 'spy_cagr': 14.02, 'a_hi': -5.3, 't_hi': -0.59, 'b_hi': 0.65, 'r2_hi': 0.859, 'a_lo': -2.5, 't_lo': -0.17, 'b_lo': 0.798, 'a_hml': -2.8, 't_hml': -0.18, 'b_hml': -0.148, 'e1_n': 80, 'e1_d': 31.2, 'e1_p': -51.7, 'e1_pt': -3.28, 'e1_h': -20.6, 'e1_ht': -1.31, 'e1_gb': 1.65, 'e2_n': 72, 'e2_d': 74.3, 'e2_p': -90.3, 'e2_pt': -3.46, 'e2_h': -15.7, 'e2_ht': -0.61, 'e2_gb': 1.22, 'q20_p': -67.6, 'q20_pt': -4.2, 'q20_h': -8.4, 'q20_ht': -0.54, 'w40_p': -65.0, 'w40_pt': -4.54, 'w40_h': -17.2, 'w40_ht': -1.23, 'ng_p': -57.0, 'ng_pt': -2.79, 'ng_h': -5.4, 'ng_ht': -0.26, 'dn_p': -68.0, 'dn_pt': -5.21, 'dn_h': -20.4, 'dn_ht': -1.61, 'core_n': 74, 'core_d': 58.1, 'core_p': -50.8, 'core_pt': -1.93, 'core_h': 7.2, 'core_ht': 0.3, 'to_hi': 0.049, 'to_lo': 0.043, 'c0': -18.3, 'c0_t': -1.24, 'c5b100': -27.1, 'c5b100_t': -1.84, 'c25b200': -37.3, 'c25b200_t': -2.52, 'syn_null_total': 0.3, 'syn_null_total_t': 0.1, 'syn_null_price': -28.8, 'syn_null_price_t': -8.8, 'syn_null_gb': 0.99, 'syn_plant_total': 30.2, 'syn_plant_total_t': 9.2, 'syn_planted': 30.0, 'syn_seeds_mean': -2.4, 'syn_seeds_sd': 3.1, 'qyld_p': -2.58, 'qyld_t': 8.41, 'ryld_p': -6.16, 'ryld_t': 5.47, 'pff_p': -2.55, 'pff_t': 3.69, 'schd_p': 9.39, 'schd_t': 12.93, 'nobl_p': 7.94, 'nobl_t': 10.21}

## Design, and the one lag

Ranking variable: the **trailing-12-month distribution rate**, reconstructed as the compounded product of `(1+r_total)/(1+r_price) − 1`. It is a **PROXY** for the marketed sticker — realised trailing rather than last-payment-annualised, and it includes capital-gains distributions.

The rank is formed at the close of month *t* and earns month *t+1*'s return. That is the **only** execution lag in the study. Everything is measured on simple monthly returns; long legs are excess-of-cash (minus BIL); the self-financing spread pays borrow (an **ASSUMPTION**, swept).

## Fama-MacBeth — three left-hand sides, one right-hand side

Slopes in bps/month per 1 sd of cross-sectionally z-scored payout rate; time-series mean over 152 months, Newey-West 6 lags.

In [2]:
for lab, k, t in [('next payout      ', 'fm_dist', 'fm_dist_t'),
                  ('price-only return', 'fm_price', 'fm_price_t'),
                  ('TOTAL return     ', 'fm_total', 'fm_total_t')]:
    print(f"{lab}: {R[k]:+7.1f} bps/mo/sd   HAC t = {R[t]:+6.2f}")

next payout      :   +24.6 bps/mo/sd   HAC t = +11.27
price-only return:   -28.1 bps/mo/sd   HAC t =  -3.84
TOTAL return     :    -3.5 bps/mo/sd   HAC t =  -0.48


## The tercile spread, decomposed

High minus low, equal-weight, monthly rebalance. The three rows do not just *happen* to add up — `total = price + payout` is how the payout was measured in the first place, so exactly two of the three rows carry information and the third is their difference. The give-back ratio is the same statement again: `give-back = 1 − hml_total/hml_payout`, so its distance from 1.00 is the total-return leg, *t* = -1.24.

In [3]:
print(f"trailing payout at formation: hi {R['dist_hi']:.2f}%  lo {R['dist_lo']:.2f}%  "
      f"spread {R['dist_spread']:.2f} pp")
print(f"payout leg : {R['hml_d']:+7.1f} bps/mo  HAC t {R['hml_d_t']:+6.2f}  "
      f"boot CI [{R['hml_d_lo']:+.1f}, {R['hml_d_hi']:+.1f}]")
print(f"price  leg : {R['hml_p']:+7.1f} bps/mo  HAC t {R['hml_p_t']:+6.2f}  "
      f"boot CI [{R['hml_p_lo']:+.1f}, {R['hml_p_hi']:+.1f}]  <- wholly below zero")
print(f"TOTAL      : {R['hml']:+7.1f} bps/mo  HAC t {R['hml_t']:+6.2f}  "
      f"boot CI [{R['hml_lo']:+.1f}, {R['hml_hi']:+.1f}]  P(>0)={R['hml_pgt0']:.3f}")
print(f"\ngive-back ratio -price/payout = {R['giveback']:.2f}  "
      f"(1.00 = a clean wash; the gap from 1.00 IS the total leg, t={R['hml_t']:+.2f})")
print(f"identity   : hml_price = hml_total - hml_payout -> "
      f"{R['hml'] - R['hml_d']:+.1f} vs {R['hml_p']:+.1f} bps, corr {R['ident_corr']:.5f}")
print(f"the TOTAL null in annual terms: [{R['hml_lo_ann']:+.2f}%, {R['hml_hi_ann']:+.2f}%] per year")

trailing payout at formation: hi 9.92%  lo 2.87%  spread 7.05 pp
payout leg :   +51.6 bps/mo  HAC t +10.27  boot CI [+42.4, +60.5]
price  leg :   -69.9 bps/mo  HAC t  -4.53  boot CI [-100.5, -40.4]  <- wholly below zero
TOTAL      :   -18.3 bps/mo  HAC t  -1.24  boot CI [-47.2, +10.4]  P(>0)=0.102

give-back ratio -price/payout = 1.36  (1.00 = a clean wash; the gap from 1.00 IS the total leg, t=-1.24)
identity   : hml_price = hml_total - hml_payout -> -69.9 vs -69.9 bps, corr 0.99995
the TOTAL null in annual terms: [-5.67%, +1.25%] per year


> 💡 *In plain words:* the fat payers hand out an extra 52 bps a month and lose an extra 70 bps of quoted price doing it. Whatever is left over is statistical noise — but noise with a wide interval: the total-return leg's CI spans [-5.67%, +1.25%] a year, so the tape kills the marketed *positive* reading and cannot exclude a real negative one.

## The excess-of-cash race and the CAPM control

The high-payout cohort is structurally lower-beta (buy-write wrappers cap their upside), so a raw spread in a bull decade is a beta bet until β is removed.

In [4]:
print(f"{'arm':6s}{'exSharpe':>10s}{'mean bps':>10s}{'vol':>8s}{'maxDD':>8s}{'CAGR':>9s}")
for tag, s, m, v, dd, c in [('hi', R['hi_sharpe'], R['hi_mean'], R['hi_vol'], R['hi_dd'], R['hi_cagr']),
                            ('lo', R['lo_sharpe'], R['lo_mean'], R['lo_vol'], R['lo_dd'], R['lo_cagr']),
                            ('SPY', R['spy_sharpe'], R['spy_mean'], R['spy_vol'], R['spy_dd'], R['spy_cagr'])]:
    print(f'{tag:6s}{s:+10.3f}{m:+10.1f}{v:7.1f}%{dd:7.1f}%{c:+8.2f}%')
print()
print(f"CAPM hi : alpha {R['a_hi']:+5.1f} bps (t {R['t_hi']:+.2f})  beta {R['b_hi']:+.3f}  R2 {R['r2_hi']:.3f}")
print(f"CAPM lo : alpha {R['a_lo']:+5.1f} bps (t {R['t_lo']:+.2f})  beta {R['b_lo']:+.3f}")
print(f"CAPM hml: alpha {R['a_hml']:+5.1f} bps (t {R['t_hml']:+.2f})  beta {R['b_hml']:+.3f}")

arm     exSharpe  mean bps     vol   maxDD     CAGR
hi        +0.737     +62.6   10.2%  -20.1%   +9.06%
lo        +0.731     +80.9   13.3%  -23.3%  +11.07%
SPY       +0.862    +104.4   14.5%  -24.4%  +14.02%

CAPM hi : alpha  -5.3 bps (t -0.59)  beta +0.650  R2 0.859
CAPM lo : alpha  -2.5 bps (t -0.17)  beta +0.798
CAPM hml: alpha  -2.8 bps (t -0.18)  beta -0.148


## Era cut (split 2020-06-30 — the income-ETF boom line)

A mechanical identity should hold in **both** halves. It does: the erosion clears |*t*| = 3 twice, and roughly doubles as the payouts themselves doubled. The total-return leg is dead in both.

In [5]:
print(f"2013-11..2020-06 (n={R['e1_n']}): payout {R['e1_d']:+6.1f}  price {R['e1_p']:+7.1f} "
      f"(t {R['e1_pt']:+5.2f})  TOTAL {R['e1_h']:+7.1f} (t {R['e1_ht']:+5.2f})  give-back {R['e1_gb']:.2f}")
print(f"2020-07..2026-06 (n={R['e2_n']}): payout {R['e2_d']:+6.1f}  price {R['e2_p']:+7.1f} "
      f"(t {R['e2_pt']:+5.2f})  TOTAL {R['e2_h']:+7.1f} (t {R['e2_ht']:+5.2f})  give-back {R['e2_gb']:.2f}")

2013-11..2020-06 (n=80): payout  +31.2  price   -51.7 (t -3.28)  TOTAL   -20.6 (t -1.31)  give-back 1.65
2020-07..2026-06 (n=72): payout  +74.3  price   -90.3 (t -3.46)  TOTAL   -15.7 (t -0.61)  give-back 1.22


## Robustness grid — sort width, the corporate-action guard, the universe

The **guard** is an ASSUMPTION *and a hindsight filter*: fund-months with |total return| above 0.50 are dropped as unadjusted corporate actions, which means the filter reads the return of the month being predicted — a fund leaves the sort formed at *t* because of its *t+1* print. No live trader could run it. Exactly one fund-month fires — NUSI's 2025-02-18 1-for-2 reverse split, which Yahoo! applied to neither tape. The **no-guard row is the live-tradable read**, and the panel is also re-run with NUSI deleted outright (no hindsight anywhere).

In [6]:
print(f"{'variant':26s}{'price (t)':>20s}{'TOTAL (t)':>20s}")
rows = [('tercile 1/3 (headline)', R['hml_p'], R['hml_p_t'], R['hml'], R['hml_t']),
        ('quintile-ish 0.20',      R['q20_p'], R['q20_pt'], R['q20_h'], R['q20_ht']),
        ('wide sort 0.40',         R['w40_p'], R['w40_pt'], R['w40_h'], R['w40_ht']),
        ('no guard (live read)',   R['ng_p'],  R['ng_pt'],  R['ng_h'],  R['ng_ht']),
        ('NUSI dropped outright',  R['dn_p'],  R['dn_pt'],  R['dn_h'],  R['dn_ht']),
        ('option-income only',     R['core_p'], R['core_pt'], R['core_h'], R['core_ht'])]
for n, p, pt, h, ht in rows:
    print(f'{n:26s}{p:+13.1f} ({pt:+5.2f}){h:+13.1f} ({ht:+5.2f})')
print()
print('The erosion survives every cut but the last: inside the nine option-income')
print(f"wrappers alone ({R['core_n']} common months) it softens to t={R['core_pt']:+.2f} and the total")
print('leg turns insignificantly positive. Part of the headline magnitude is')
print('cross-cohort (buy-write vs dividend-equity), not a within-cohort law.')

variant                              price (t)           TOTAL (t)
tercile 1/3 (headline)            -69.9 (-4.53)        -18.3 (-1.24)
quintile-ish 0.20                 -67.6 (-4.20)         -8.4 (-0.54)
wide sort 0.40                    -65.0 (-4.54)        -17.2 (-1.23)
no guard (live read)              -57.0 (-2.79)         -5.4 (-0.26)
NUSI dropped outright             -68.0 (-5.21)        -20.4 (-1.61)
option-income only                -50.8 (-1.93)         +7.2 (+0.30)

The erosion survives every cut but the last: inside the nine option-income
wrappers alone (74 common months) it softens to t=-1.93 and the total
leg turns insignificantly positive. Part of the headline magnitude is
cross-cohort (buy-write vs dividend-equity), not a within-cohort law.


## Cost × borrow sweep on the tradable leg

Turnover is tiny, so friction is not what kills this — the spread is already insignificant gross. Borrow is an ASSUMPTION (the tape carries none).

In [7]:
print(f"one-way turnover/mo: hi {R['to_hi']:.3f}  lo {R['to_lo']:.3f}")
print(f"gross              : {R['c0']:+7.1f} bps/mo (t {R['c0_t']:+.2f})")
print(f"5 bps + 100 bps/yr : {R['c5b100']:+7.1f} bps/mo (t {R['c5b100_t']:+.2f})")
print(f"25 bps + 200 bps/yr: {R['c25b200']:+7.1f} bps/mo (t {R['c25b200_t']:+.2f})")
print('\nFriction only deepens a spread that was never positive. There is no')
print('cost assumption at which ranking on payout becomes a total-return edge.')

one-way turnover/mo: hi 0.049  lo 0.043
gross              :   -18.3 bps/mo (t -1.24)
5 bps + 100 bps/yr :   -27.1 bps/mo (t -1.84)
25 bps + 200 bps/yr:   -37.3 bps/mo (t -2.52)

Friction only deepens a spread that was never positive. There is no
cost assumption at which ranking on payout becomes a total-return edge.


## Live synthetic control — **offline simulation, not the real tape**

Three worlds. The **null**: the payout is pure return of capital — erosion must fire, total return must not. The **planted** world: a genuine one-for-one yield-to-return bonus — the total leg must fire and land on the planted value. The **beta confound**: no alpha, but beta falls across the yield sort — the raw spread goes negative and the CAPM control must absorb it.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from dist_illusion import data, strategy as st
null,  _     = data.synthetic_panel(signal_strength=0.0, seed=946)
plant, truth = data.synthetic_panel(signal_strength=1.0, seed=946)
conf,  _     = data.synthetic_panel(signal_strength=0.0, beta_slope=0.5, seed=946)
dn, dp, dc = st.synthetic_detect(null), st.synthetic_detect(plant), st.synthetic_detect(conf)
print('SYNTHETIC (machinery proof, never supports the real-tape stamp)')
print('  null    : total %+6.1f (t %+6.2f)  price %+6.1f (t %+6.2f)  give-back %.2f'
      % (dn['fm_total_bps'], dn['t_total'], dn['fm_price_bps'], dn['t_price'], dn['giveback']))
print('  planted : total %+6.1f (t %+6.2f)  [planted %+6.1f]'
      % (dp['fm_total_bps'], dp['t_total'], truth['planted_slope_per_sd']*1e4))
print('  beta cfd: raw HML %+6.1f (t %+.2f) -> CAPM alpha %+.1f (t %+.2f), beta %+.3f'
      % (dc['hml_bps'], dc['t_hml'], dc['capm_hml']['alpha_bps'],
         dc['capm_hml']['t_alpha'], dc['capm_hml']['beta']))
seeds = np.array([st.synthetic_detect(data.synthetic_panel(signal_strength=0.0, seed=946+s)[0])['fm_total_bps']
                  for s in range(8)])
print('  null across 8 seeds: mean %+.1f bps, sd %.1f, |mean|>20 on %d/8'
      % (seeds.mean(), seeds.std(ddof=1), (np.abs(seeds) > 20).sum()))

SYNTHETIC (machinery proof, never supports the real-tape stamp)
  null    : total   +0.3 (t  +0.10)  price  -28.8 (t  -8.80)  give-back 0.99
  planted : total  +30.2 (t  +9.20)  [planted  +30.0]
  beta cfd: raw HML  -18.1 (t -0.44) -> CAPM alpha -1.2 (t -0.18), beta -1.103


  null across 8 seeds: mean -2.4 bps, sd 3.1, |mean|>20 on 0/8


## Verdict

- **Signal — Real**, on two measured facts rather than three. (1) The payout rank forecasts the **next payout**: *t* = +11.27, era-stable, and the only estimate here that restates nothing else. (2) It is **uninformative about total return**: -18.3 bps/mo, *t* = -1.24, CI [-5.67%, +1.25%] a year; CAPM α -2.8 bps, *t* = -0.18. The **NAV erosion** (-69.9 bps/mo, HAC *t* = -4.53, CI [-100.5, -40.4], |*t*| > 3 in both eras, stable across sort widths, guard thresholds and with NUSI deleted) is **the identity of those two**, reported as arithmetic and never as independent confirmation. Give-back **1.36** — i.e. 1.00 plus an insignificant total leg, so more-than-one-for-one erosion is a point estimate, not a finding. Named limits: the erosion is partly cross-cohort (*t* = -1.93 within the nine option-income wrappers over 74 months), the sample is **survivorship-selected** (and the sort skips funds without a next-month print), the ranking variable is a **PROXY**, and the corporate-action guard is a hindsight filter whose no-guard alternative reads -57.0 (*t* = -2.79).
- **Tradability — Mirage.** The self-financing expression (high minus low) is -18.3 bps/mo gross (*t* = -1.24) — i.e. +18.3 bps if you flip it long-low/short-high — CI straddling zero, and the short leg pays borrow it cannot afford. The long-only expression is a 0.65-beta clone with α = -5.3 bps (*t* = -0.59) that compounds at 9.06%/yr against SPY's 14.02%. Nothing to bank in either direction.